# Track 2 · Stage 1 — Generate the code-mixed **text**

Replication of Biswas et al., Interspeech 2025 (Track 2).

Few-shot-prompt **Gemma 4 26B-A4B** (served by **llama.cpp**) for Hindi-English bigrams →
filter them → expand each into four sentences (~16k). Push the text to
`RohanRamesh/hi-en-synth-cs`.

Audio synthesis is a **separate notebook** (`01b`). It needs `transformers==4.46.1` for
parler-tts; the LLM stage here needs neither that pin nor transformers at all (llama.cpp
uses the GGUF's own chat template). The Hub is the checkpoint between them.

### The model: `unsloth/gemma-4-26B-A4B-it-GGUF`
* **apache-2.0, ungated.** A 25.2B-total / **3.8B-active** MoE — far stronger than the dense
  E4B used earlier, which produced text that was too English-heavy and switched too little
  (deviation D11). This should fix that.
* Served by **llama.cpp** (`backend: llamacpp`), Q4_K_M ≈ 16 GB. That does **not** fit one
  T4, so `tensor_split=[0.5, 0.5]` spreads the single model across **both** cards.
* **Consequences, stated plainly:**
  - *One process, both GPUs.* No cross-GPU sharding here — both T4s hold the one model. The
    per-GPU sharding used for the dense E4B does not apply.
  - *Single-stream and slow.* llama.cpp chat completions are sequential, so the full run is
    ~8–12 h. Every call is cached, so a 12 h session cap is survived by re-running.
  - *Thinking is disabled* (`disable_thinking: true`) — reasoning tokens would multiply the
    runtime. The backend tries `enable_thinking=False` and strips any reasoning that leaks;
    the smoke test flags it if thinking survives.

### Before you run
* HF **write** token in Kaggle Secrets as `HF_TOKEN`. Internet **on**, **GPU T4 ×2**.
* Nothing to accept — the model is apache-2.0. (Parler-TTS *is* gated; that bites in `01b`.)

Every LLM call is cached, so a 12 h timeout costs nothing on a re-run.

## 0 · Install

The cu121 **prebuilt** llama-cpp-python wheel (with CUDA offload) — no 15-minute compile. NO transformers/parler-tts here.

In [4]:
# Prebuilt CUDA wheel: ships with GPU offload, avoids a long source build.
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install -q llama-cpp-python -U --force-reinstall --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install -q "datasets<4" librosa soundfile soxr omegaconf rich huggingface_hub
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

NOTEBOOK_VERSION = "0.10.0"

import csasr
assert csasr.__version__ == NOTEBOOK_VERSION, (
    f"csasr package is {csasr.__version__} but this NOTEBOOK is {NOTEBOOK_VERSION}.\n"
    "  package older  -> restart the kernel (Run > Restart & clear); pip skips a\n"
    "                    reinstall when the version looks satisfied.\n"
    "  notebook older -> re-download it from the repo; pip does NOT update .ipynb files."
)

import llama_cpp
print("csasr", csasr.__version__, "| llama_cpp", llama_cpp.__version__,
      "| GPU offload:", llama_cpp.llama_supports_gpu_offload())
assert llama_cpp.llama_supports_gpu_offload(), (
    "llama.cpp has no GPU offload -- the CPU-only wheel got installed. "
    "Re-run the cu121 --extra-index-url line above."
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 594.3 kB/s eta 0:00:000:01m00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
csasr 0.10.0 requires jiwer>=3.0, which is not installed.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires de

In [5]:
import os, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
HF_TOKEN = os.environ["HF_TOKEN"]

REAL_REPO  = "RohanRamesh/mucs-he-cs"
SYNTH_REPO = "RohanRamesh/hi-en-synth-cs"
# llama.cpp serves this GGUF split across BOTH T4s (tensor_split). One process,
# both GPUs -- so no per-GPU sharding here (unlike the old dense-model notebook).
LLM  = "unsloth/gemma-4-26B-A4B-it-GGUF"
GGUF = "gemma-4-26B-A4B-it-UD-Q4_K_M.gguf"

WORK  = Path("/kaggle/working")
MAN   = WORK / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    p = subprocess.run([sys.executable, "-m", *args])   # inherits HF_TOKEN + streams
    if p.returncode != 0:
        raise RuntimeError(
            f"{args[0]} failed (exit {p.returncode}). The real error is printed ABOVE "
            f"this traceback - scroll up in this cell's output."
        )

def gen(module, *args):
    """Run one LLM stage. The model spans both GPUs via tensor_split, so this is
    a single subprocess -- NOT sharded. Cache makes it resumable across sessions."""
    run(module, *args, "--backend", "llamacpp", "--model", LLM, "--gguf-file", GGUF)

from csasr.manifest import read_jsonl, write_jsonl

import torch
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}  {free/2**30:5.1f} GiB free / {total/2**30:.1f} GiB")
!nvidia-smi --query-gpu=name,memory.total --format=csv

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


cuda:0   14.5 GiB free / 14.6 GiB
cuda:1   14.5 GiB free / 14.6 GiB
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 1 · Pull the in-domain transcripts (few-shot exemplars)

Track 2 never trains on real code-switched audio — we only need the *text* of the MUCS train split.

In [6]:
from datasets import load_dataset
from csasr.manifest import write_jsonl

train_text = load_dataset(REAL_REPO, "train_text", split="train", token=HF_TOKEN)
write_jsonl(MAN / "mucs_train.jsonl", [dict(r) for r in train_text])
print(f"{len(train_text):,} in-domain sentences for few-shot prompting")
print(train_text[0]["text"])

52,825 in-domain sentences for few-shot prompting
दोस्तों bash में nested और multilevel if statement के spoken tutorial में आपका स्वागत है


## 2 · SMOKE TEST — load the model, generate 20 bigrams

Loads the 26B GGUF across both T4s, generates real bigrams, and shows which survive the
script filter — in a few minutes, before committing to the ~8–12 h full run.

**It runs as a subprocess**, so the model is freed cleanly before the real stages start
(Jupyter's `Out[]` history would otherwise pin the VRAM).

Two things to check in the output:
* **thinking is off** — if the smoke test warns that `<think>`/`<|channel|>` markers survived,
  stop and tell the maintainer;
* Gemma often emits **three**-word phrases (`बुनियादी formatting basics`); the filter extracts
  the switch pair from inside them (deviation **D9**), so that is fine.

In [7]:
run("csasr.llm.smoke", "--backend", "llamacpp", "--model", LLM, "--gguf-file", GGUF,
    "--train-manifest", MAN / "mucs_train.jsonl",
    "--n-calls", "2", "--bigrams-per-call", "10")

> csasr.llm.smoke --backend llamacpp --model unsloth/gemma-4-26B-A4B-it-GGUF --gguf-file gemma-4-26B-A4B-it-UD-Q4_K_M.gguf --train-manifest /kaggle/working/manifests/mucs_train.jsonl --n-calls 2 --bigrams-per-call 10


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
smoke (cached 0):   0%|          | 0/2 [00:00<?, ?req/s]

[smoke] model  : unsloth/gemma-4-26B-A4B-it-GGUF
[smoke] prompts: 2 x 10 bigrams

[llm] llama.cpp loading unsloth/gemma-4-26B-A4B-it-GGUF/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf
[llm]   n_gpu_layers=-1  tensor_split=[0.5, 0.5]  n_ctx=4096


smoke (cached 0): 100%|██████████| 2/2 [00:02<00:00,  1.19s/req]


## 3 · Generate bigrams

Paper: 44,657 raw → 5,932 unique (13.3%). Single process, both GPUs, cached — safe to re-run after a session timeout.

In [8]:
gen("csasr.llm.gen_bigrams",
    "--train-manifest", MAN / "mucs_train.jsonl",
    "--out", MAN / "bigrams_raw.jsonl",
    "--cache", CACHE / "bigrams.jsonl",
    "--n-calls", "4466")

> csasr.llm.gen_bigrams --train-manifest /kaggle/working/manifests/mucs_train.jsonl --out /kaggle/working/manifests/bigrams_raw.jsonl --cache /kaggle/working/llm_cache/bigrams.jsonl --n-calls 4466 --backend llamacpp --model unsloth/gemma-4-26B-A4B-it-GGUF --gguf-file gemma-4-26B-A4B-it-UD-Q4_K_M.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
bigrams (cached 0): 100%|██████████| 4466/4466 [1:07:47<00:00,  1.10req/s]


## 4 · Filter

Deterministic script filter (one Devanagari token + one Latin token), then an LLM translation check with 3-sample self-consistency.
Paper: 5,932 unique → 5,477 valid (92.3%).

In [9]:
gen("csasr.llm.filter_bigrams",
    "--raw", MAN / "bigrams_raw.jsonl",
    "--out", MAN / "bigrams_valid.jsonl",
    "--cache", CACHE / "transcheck.jsonl",
    "--items-per-call", "20", "--n-samples", "3")

> csasr.llm.filter_bigrams --raw /kaggle/working/manifests/bigrams_raw.jsonl --out /kaggle/working/manifests/bigrams_valid.jsonl --cache /kaggle/working/llm_cache/transcheck.jsonl --items-per-call 20 --n-samples 3 --backend llamacpp --model unsloth/gemma-4-26B-A4B-it-GGUF --gguf-file gemma-4-26B-A4B-it-UD-Q4_K_M.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
transcheck 3/3 (cached 0): 100%|██████████| 169/169 [05:55<00:00,  2.10s/req]


## 5 · Expand each bigram into four sentences

2 English-matrix, 2 Hindi-matrix. Paper: ~16,000 unique from a theoretical 21,908.

In [10]:
gen("csasr.llm.gen_sentences",
    "--bigrams", MAN / "bigrams_valid.jsonl",
    "--out", MAN / "sentences.jsonl",
    "--cache", CACHE / "sentences.jsonl")

> csasr.llm.gen_sentences --bigrams /kaggle/working/manifests/bigrams_valid.jsonl --out /kaggle/working/manifests/sentences.jsonl --cache /kaggle/working/llm_cache/sentences.jsonl --backend llamacpp --model unsloth/gemma-4-26B-A4B-it-GGUF --gguf-file gemma-4-26B-A4B-it-UD-Q4_K_M.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
sentences (cached 0): 100%|██████████| 2947/2947 [1:00:33<00:00,  1.23s/req]


### GATE 1 — yields must track the paper

A large divergence in the 13.3% dedup rate means the prompt or temperature is off. Decide here, not after 6 hours of TTS.

In [11]:
raw   = list(read_jsonl(MAN / "bigrams_raw.jsonl"))
uniq  = {r["bigram"] for r in raw}
valid = list(read_jsonl(MAN / "bigrams_valid.jsonl"))
sents = list(read_jsonl(MAN / "sentences.jsonl"))

rows = [
    ("raw bigrams",    len(raw),   44_657, None),
    ("unique bigrams", len(uniq),   5_932, len(uniq) / max(len(raw), 1)),
    ("valid bigrams",  len(valid),  5_477, len(valid) / max(len(uniq), 1)),
    ("sentences",      len(sents), 16_000, None),
]
print(f"{'metric':<16}{'ours':>10}{'paper':>10}{'survival':>12}")
for name, got, want, surv in rows:
    s = f"{surv:.1%}" if surv else "-"
    print(f"{name:<16}{got:>10,}{want:>10,}{s:>12}")
print("\npaper survival: dedup 13.3%, filter 92.3%")
print("\nGemma 4 26B-A4B is smaller than the paper's 70B (deviation D1), so a lower")
print("valid-bigram yield is expected. What matters is that ENOUGH sentences survive:")
print(f"  -> {len(sents):,} sentences  (need >~8,000 for a usable 22h corpus)")

for r in sents[:5]:
    print("   ", r["text"])

metric                ours     paper    survival
raw bigrams         44,713    44,657           -
unique bigrams       3,458     5,932        7.7%
valid bigrams        2,947     5,477       85.2%
sentences            9,031    16,000           -

paper survival: dedup 13.3%, filter 92.3%

Gemma 4 26B-A4B is smaller than the paper's 70B (deviation D1), so a lower
valid-bigram yield is expected. What matters is that ENOUGH sentences survive:
  -> 9,031 sentences  (need >~8,000 for a usable 22h corpus)
    Please ensure that you check the Content को carefully before publishing it on the website.
    I am working hard to improve the Content को so that our audience finds it more engaging.
    हमें अपने Content को और भी बेहतर बनाने की ज़रूरत है।
    आप अपने Content को सोशल मीडिया पर कैसे शेयर करते हैं?
    You should consider this option पर ध्यान देना चाहिए before making a final decision.


### Repair — strip the matrix-language labels

Gemma prefixes each sentence with its matrix language:

    English: Many software programs have different aliases निर्धारित for commands.

Left in, Parler-TTS would literally **speak** "English colon, many software programs…" and
Whisper would then be **trained to emit `English:`** at the start of every transcript. This
re-cleans the text, re-checks that the bigram survived, and recomputes the matrix language
(the prefix biased it). Seconds — no LLM re-run.

In [12]:
run("csasr.llm.fix_sentences",
    "--in", MAN / "sentences.jsonl",
    "--bigrams", MAN / "bigrams_valid.jsonl",
    "--out", MAN / "sentences.jsonl")

sents = list(read_jsonl(MAN / "sentences.jsonl"))
print(f"\n{len(sents):,} sentences ready for TTS:")
for r in sents[:5]:
    print("   ", r["text"])

> csasr.llm.fix_sentences --in /kaggle/working/manifests/sentences.jsonl --bigrams /kaggle/working/manifests/bigrams_valid.jsonl --out /kaggle/working/manifests/sentences.jsonl

9,031 sentences ready for TTS:
    Please ensure that you check the Content को carefully before publishing it on the website.
    I am working hard to improve the Content को so that our audience finds it more engaging.
    हमें अपने Content को और भी बेहतर बनाने की ज़रूरत है।
    आप अपने Content को सोशल मीडिया पर कैसे शेयर करते हैं?
    You should consider this option पर ध्यान देना चाहिए before making a final decision.


## 6 · Push the text to the Hub

This is the handoff to `01b`. Push before anything can time out.

In [13]:
for man, cfg in [("bigrams_valid.jsonl", "bigrams"), ("sentences.jsonl", "sentences")]:
    run("csasr.data.push_to_hub", "--manifest", MAN / man,
        "--repo", SYNTH_REPO, "--config", cfg, "--text-only")
print("\ntext stage complete -> now run 01b_synthesize_audio.ipynb")

> csasr.data.push_to_hub --manifest /kaggle/working/manifests/bigrams_valid.jsonl --repo RohanRamesh/hi-en-synth-cs --config bigrams --text-only


Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 1042.67ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|██████████| 58.3kB / 58.3kB            

Processing Files (1 / 1)      : 100%|██████████| 58.3kB / 58.3kB, 76.1kB/s  

                              : 100%|██████████| 58.3kB / 58.3kB            

                              : 100%|██████████| 58.3kB / 58.3kB            

                              : 100%|██████████| 58.3kB / 58.3kB            

                              : 100%|██████████| 58.3kB / 58.3kB            

Processing Files (1 / 1)      : 100%|██████████| 58.3kB / 58.3kB, 37.2kB/s  
New Data Upload               : |          |  0.00B /  0.00B,  0.00B/s  
                              : 100%|██████████| 58.3kB / 58.3kB            
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.92s/it

> csasr.data.push_to_hub --manifest /kaggle/working/manifests/sentences.jsonl --repo RohanRamesh/hi-en-synth-cs --config sentences --text-only


Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 1092.49ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|██████████|  642kB /  642kB            

Processing Files (1 / 1)      : 100%|██████████|  642kB /  642kB,  842kB/s  

                              : 100%|██████████|  642kB /  642kB            

                              : 100%|██████████|  642kB /  642kB            

                              : 100%|██████████|  642kB /  642kB            

                              : 100%|██████████|  642kB /  642kB            

                              : 100%|██████████|  642kB /  642kB            

Processing Files (1 / 1)      : 100%|██████████|  642kB /  642kB,  364kB/s  
New Data Upload               : |          |  0.00B /  0.00B,  0.00B/s  
                              : 100%|██████████|  642kB /  642kB       


text stage complete -> now run 01b_synthesize_audio.ipynb
